# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [11]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
from typing import Optional, Literal
from IPython.display import display, Markdown

In [12]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [18]:
# set up environment
load_dotenv()


# OLLAMA_BASE_URL = "http://localhost:11434/v1"
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

OLLAMA_BASE_URL = "http://localhost:11434/v1"

class Model:
    # def __init__(self):
    #     self.client_oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    #     # self.ollama_base_url = f"{os.getenv('OLLAMA_URL')}/api/chat"
    #     self.ollama=OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

    def __init__(self):
        # Use the URL string from env, strip quotes/whitespace and ensure no trailing slash
        ollama_url = os.getenv("OLLAMA_URL")
        if not ollama_url:
            raise RuntimeError("OLLAMA_URL not set in environment")
        ollama_url = ollama_url.strip().strip('"').rstrip('/')
        # Ollama chat API endpoint
        self.ollama = f"{ollama_url}/api/chat"
        self.client_oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))




    def _prompt_llama(self, text: str):
        response = requests.post(
            self.ollama,
            json={
                "model": MODEL_LLAMA,
                "messages": [{"role": "user", "content": text}],
                "stream": True
            },
            stream=True
        )
        for line in response.iter_lines():
            if not line:
                continue
                
            data = line.decode("utf-8")
            if data.strip() == "":
                continue
                
            if data.startswith("data:"):
                data = data[5:].strip()
                
            yield data

    def _prompt_oai(self, question: str):
        stream = self.client_oai.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an advanced reasoning and explanation engine. "
                        "You write with precision, clarity, and conciseness. "
                        "You can explain Python, algorithms, code design, and systems-level behavior "
                        "with technical rigor, while being straight to the point."
                    ),
                },
                {"role": "user", "content": question},
            ],
            stream=True,
        )
        return stream

    def prompt(self, question: str, model: Optional[Literal[MODEL_GPT, MODEL_LLAMA]] = MODEL_GPT):
        if "gpt" in model:
            stream = self._prompt_oai(question)
            buffer = []
            for event in stream:
                if event.choices and event.choices[0].delta.content:
                    chunk = event.choices[0].delta.content
                    buffer.append(chunk)
                    yield chunk
                    
            output = "".join(buffer)

        else:
            stream = self._prompt_llama(question)
            buffer = []
            for chunk in stream:
                try:
                    import json
                    data = json.loads(chunk)
                    content = data.get("message", {}).get("content", "")
                    
                    if content:
                        buffer.append(content)
                        yield content
                        
                except Exception as e:
                    print("An error occured", e)
                    continue
                    
            output = "".join(buffer)

        display(Markdown(output))


In [14]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [15]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [21]:
# The question for Ollama

question_Ollama = """ 
Please explain what this code does and why:

if __name__ == "__main__"

"""

In [20]:
# Get gpt-4o-mini to answer, with streaming
model = Model()

for token in model.prompt(question, model=MODEL_GPT):
    print(token, end="", flush=True)


This code snippet utilizes a Python generator to yield values from a set comprehension. Here's a breakdown of its components:

1. **Set Comprehension**: 
   - `{book.get("author") for book in books if book.get("author")}` creates a set of authors from a list of `books`.
   - `book.get("author")` retrieves the "author" field from each `book` dictionary.
   - The condition `if book.get("author")` ensures that only books with a non-None and non-empty "author" field are included.

2. **`yield from` Statement**:
   - `yield from` is a way to yield all values from an iterable (in this case, the set created by the comprehension) one at a time. 
   - This means that instead of returning the entire set at once, the generator will yield each author individually as it is iterated over.

### Purpose:
The code functions as a generator that provides unique author names from the `books` collection. It ensures that only authors who are actually present (i.e., the "author" key is valid) are yielded. Th

This code snippet utilizes a Python generator to yield values from a set comprehension. Here's a breakdown of its components:

1. **Set Comprehension**: 
   - `{book.get("author") for book in books if book.get("author")}` creates a set of authors from a list of `books`.
   - `book.get("author")` retrieves the "author" field from each `book` dictionary.
   - The condition `if book.get("author")` ensures that only books with a non-None and non-empty "author" field are included.

2. **`yield from` Statement**:
   - `yield from` is a way to yield all values from an iterable (in this case, the set created by the comprehension) one at a time. 
   - This means that instead of returning the entire set at once, the generator will yield each author individually as it is iterated over.

### Purpose:
The code functions as a generator that provides unique author names from the `books` collection. It ensures that only authors who are actually present (i.e., the "author" key is valid) are yielded. This might be useful for creating a list of unique authors from a larger collection of books, effectively filtering out any entries where the author is not specified.

### Example Context:
If `books` is a list of dictionaries such as:

```python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": None},
    {"title": "Book C", "author": "Author 2"},
    {"title": "Book D", "author": "Author 1"}
]
```

The resulting set of unique authors would be `{'Author 1', 'Author 2'}`, and each author would be yielded one at a time when iterating over the generator produced by this expression.

In [19]:
print(os.getenv("OLLAMA_URL"))

http://localhost:11434


In [22]:
# Get Llama 3.2 to answer
model = Model()

for token in model.prompt(question_Ollama, model=MODEL_LLAMA):
    print(token, end="", flush=True)

This line of code is a common Python idiom that serves as a guard clause to determine if the script is being run directly or imported as a module.

Here's a breakdown of what it does:

- `if`: This keyword starts an if statement, which will be evaluated.
- `__name__ == "__main__"`: In Python, when a script is executed directly (i.e., not imported), the special variable `__name__` is set to `"__main__"`. However, when a script is imported as a module, `__name__` is set to the name of the module.

So, the condition checks whether `__name__` is equal to `"__main__"`.

- If the condition is true, the code within this block will be executed.

In essence, this line is used to ensure that certain code only runs when the script is executed directly, and not when it's imported as a module.

Here are some scenarios where you might use this idiom:

1. Testing: You can write test cases in a separate file, and then run them using `python -m unittest mytestmodule.py`. By putting your main code insid

This line of code is a common Python idiom that serves as a guard clause to determine if the script is being run directly or imported as a module.

Here's a breakdown of what it does:

- `if`: This keyword starts an if statement, which will be evaluated.
- `__name__ == "__main__"`: In Python, when a script is executed directly (i.e., not imported), the special variable `__name__` is set to `"__main__"`. However, when a script is imported as a module, `__name__` is set to the name of the module.

So, the condition checks whether `__name__` is equal to `"__main__"`.

- If the condition is true, the code within this block will be executed.

In essence, this line is used to ensure that certain code only runs when the script is executed directly, and not when it's imported as a module.

Here are some scenarios where you might use this idiom:

1. Testing: You can write test cases in a separate file, and then run them using `python -m unittest mytestmodule.py`. By putting your main code inside the guard clause, you ensure that the tests only run when the script is executed directly.
2. Command-line tools: If you're writing a command-line tool, you might want to put the main logic inside the guard clause, so it's only executed when the script is run from the command line.
3. Modules with setup/teardown code: If your module has some setup or teardown code that needs to be run every time it's imported, you can put it inside the guard clause.

Here's an example of how this might look in a real-world scenario:

```python
# myscript.py

if __name__ == "__main__":
    # Main logic goes here
    print("This is the main script")
else:
    # Code that should only run when the module is imported
    print("This code runs when importing the module")

def some_function():
    # Function implementation
    pass
```

In this example, `some_function()` will be executed every time the `myscript.py` module is imported, but the main logic (`print("This is the main script")`) will only run when `myscript.py` is executed directly.